In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
df = pd.read_csv('../data/ai_student_impact_dataset (1).csv')
target = "Post_Semester_GPA"
base_feats = ["Pre_Semester_GPA", "Weekly_GenAI_Hours", "Tool_Diversity"]

In [4]:
centered = df[base_feats].copy()
for col in base_feats:
    centered[col] = df[col] - df[col].mean()

In [5]:
X_base = df[base_feats].copy()

X_engineered = df[base_feats].copy()
X_engineered["Weekly_GenAI_Hours_Sq"] = centered["Weekly_GenAI_Hours"] ** 2
X_engineered["PreGPA_Sq"] = centered["Pre_Semester_GPA"] ** 2
X_engineered["PreGPA_x_Tool"] = centered["Pre_Semester_GPA"] * centered["Tool_Diversity"]
X_engineered["AI_Engagement_Index"] = centered["Weekly_GenAI_Hours"] * centered["Tool_Diversity"]

y = df[target]

X_train_b, X_test_b, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)
X_train_e, X_test_e, _, _ = train_test_split(X_engineered, y, test_size=0.2, random_state=42)

In [6]:
def evaluate_model(X_train, X_test, y_train, y_test):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return mse, mae, r2

mse_b, mae_b, r2_b = evaluate_model(X_train_b, X_test_b, y_train, y_test)
mse_e, mae_e, r2_e = evaluate_model(X_train_e, X_test_e, y_train, y_test)

In [18]:
results_df = pd.DataFrame({
    'Chỉ số Đánh giá': ['Sai số Bình phương Trung bình (MSE)', 'Sai số Tuyệt đối Trung bình (MAE)', 'Độ chính xác R-squared (R²)'],
    'Trước FE (Biến gốc)': [mse_b, mae_b, r2_b],
    'Sau FE (Biến Cải tiến)': [mse_e, mae_e, r2_e]
})

# Tính toán mức độ cải thiện
results_df['Mức cải thiện thực tế'] = results_df['Trước FE (Biến gốc)'] - results_df['Sau FE (Biến Cải tiến)']
# Riêng R2 thì càng cao càng tốt nên lấy Sau trừ Trước
results_df.loc[2, 'Mức cải thiện thực tế'] = r2_e - r2_b

print("\nSO SÁNH HIỆU SUẤT TRƯỚC VÀ SAU FEATURE ENGINEERING")
display(results_df.style.format({'Trước FE (Biến gốc)': '{:.6f}', 'Sau FE (Biến Cải tiến)': '{:.6f}', 'Mức cải thiện thực tế': '{:.6f}'}))



SO SÁNH HIỆU SUẤT TRƯỚC VÀ SAU FEATURE ENGINEERING


,Chỉ số Đánh giá,Trước FE (Biến gốc),Sau FE (Biến Cải tiến),Mức cải thiện thực tế
0,Sai số Bình phương Trung bình (MSE),0.033649,0.033029,0.000620
1,Sai số Tuyệt đối Trung bình (MAE),0.143174,0.142038,0.001135
2,Độ chính xác R-squared (R²),0.860600,0.863169,0.002569
